In [9]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../balance_metrics")

import yaml
from balance_metrics import plot_epoch_metrics_vs_accuracies, get_data, plot_layer_metrics, plot_exp_regressions, plot_linear_regressions

data_path = "../../experiment_data/balance_metrics/cross_layer_random.csv"
layer_path_better = "../../experiment_data/balance_metrics/cross_layer_better.csv"

random_data = get_data(data_path)
layer_data_better = get_data(layer_path_better)

with open('../layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)
        
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
full_data = pd.concat([random_data, layer_data_better], ignore_index=True)
full_data = full_data.sort_values(["dataset"], ascending=True)

In [11]:
resnet = full_data[(full_data["model"].str.contains("resnet")) & (full_data["split"] == "val")].copy()

resnet["layer_idx"] = resnet["layer"].apply(lambda x: layer_order['resnet'].index(x))
resnet.sort_values("layer_idx", inplace=True)

resnet_accs = resnet[(resnet["layer"] == "a1") & (resnet["dataset"].str.contains("-r"))][["dataset", "model", "train_acc", "val_acc"]]
resnet_accs["random_prop"] = resnet_accs["dataset"].apply(lambda x: float(x.split("-r")[1]))
resnet_accs.sort_values("random_prop", inplace=True)

fig = make_subplots()
for i, row in resnet_accs.iterrows():
    color = color_seq[i % len(color_seq)]
    fig.add_trace(go.Scatter(x=[row["random_prop"], row["random_prop"]], y=[row["train_acc"], row["val_acc"]], mode="lines+markers", line = dict(color=color), name=f"{row['model']} ({row['random_prop']})"))
    
fig.update_layout(title="ResNet Accuracies vs Random Label Proportion")
fig.update_xaxes(title_text="Randomness Proportion", nticks=23)
fig.update_yaxes(range=[0.0, 1.05], nticks=20)

In [12]:
fig = px.line(resnet, x="layer", y="colless_index", color="dataset", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="ResNet Colless")
fig.update_xaxes(tickangle=80)
fig.update_yaxes(type="log")

In [13]:
fig = px.line(resnet, x="layer", y="sackin_index", color="dataset", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="ResNet Sackin")
fig.update_xaxes(tickangle=80)
fig.update_yaxes(type="log")

In [14]:
densenet = full_data[(full_data["model"].str.contains("densenet")) & (full_data["split"] == "val")].copy()

densenet["layer_idx"] = densenet["layer"].apply(lambda x: layer_order['densenet'].index(x))
densenet.sort_values("layer_idx", inplace=True)

densenet_accs = densenet[(densenet["layer"] == "a1") & (densenet["dataset"].str.contains("-r"))][["dataset", "model", "train_acc", "val_acc"]]
densenet_accs["random_prop"] = densenet_accs["dataset"].apply(lambda x: float(x.split("-r")[1]))
densenet_accs.sort_values("random_prop", inplace=True)

fig = make_subplots()
for i, row in densenet_accs.iterrows():
    color = color_seq[i % len(color_seq)]
    fig.add_trace(go.Scatter(x=[row["random_prop"], row["random_prop"]], y=[row["train_acc"], row["val_acc"]], mode="lines+markers", line = dict(color=color), name=f"{row['model']} ({row['random_prop']})"))
    
fig.update_layout(title="DenseNet Accuracies vs Random Label Proportion")
fig.update_xaxes(title_text="Randomness Proportion", nticks=23)
fig.update_yaxes(range=[0.0, 1.05], nticks=20)

In [15]:
fig = px.line(densenet, x="layer", y="colless_index", color="dataset", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="densenet Colless")
fig.update_xaxes(tickangle=80)
fig.update_yaxes(type="log")

In [16]:
fig = px.line(densenet, x="layer", y="sackin_index", color="dataset", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="densenet Sackin")
fig.update_xaxes(tickangle=80)
fig.update_yaxes(type="log")